# PACE-KG — Steps 1 → 4 (Colab Pipeline)

Run the cells **in order**. Each step saves its output as a JSON file and
offers a download button so you can keep intermediate results.

| Step | What it does | Output file |
|------|-------------|-------------|
| 1 | Marker PDF parsing + OCR fallback | `*_step1_parsed.json` |
| 2 | Markdown preprocessor (bucket typing, noise removal, cross-slide filter) | `*_step2_preprocessed.json` |
| 3 | SIFRank-style keyphrase extraction | `*_step3_keyphrases.json` |
| 4 | LLM triple extraction + 3-layer validation | `*_step4_triples.json` |

> **GPU recommended** for Step 3 (SBERT) and Step 4 (SBERT + LLM calls).  
> Runtime → Change runtime type → T4 GPU.

## Cell 0 — Install dependencies
Run once per Colab session.

In [ ]:
# ── Core parsing deps ──────────────────────────────────────────────────────
!pip install marker-pdf -q
!pip install pytesseract pillow pymupdf -q
!apt-get install -y tesseract-ocr -q

# ── NLP / ML deps ──────────────────────────────────────────────────────────
!pip install sentence-transformers spacy -q
!python -m spacy download en_core_web_sm -q

# ── LLM / LangChain deps ───────────────────────────────────────────────────
!pip install langchain langchain-groq -q

# ── Quick sanity check ─────────────────────────────────────────────────────
import importlib, subprocess, sys
for pkg in ["marker", "pytesseract", "fitz", "PIL",
            "sentence_transformers", "spacy", "langchain_groq"]:
    ok = importlib.util.find_spec(pkg) is not None
    print(f"{'OK' if ok else 'MISSING':6s}  {pkg}")

res = subprocess.run(["tesseract", "--version"], capture_output=True, text=True)
print(f"{'OK' if res.returncode == 0 else 'MISSING':6s}  tesseract",
      res.stdout.splitlines()[0] if res.returncode == 0 else "")

print("\nAll dependencies installed.")

## Cell 1 — Upload PDF
Upload the lecture slide PDF you want to process.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
DOC_ID   = Path(PDF_PATH).stem          # used as document identifier throughout
STEM     = DOC_ID                       # base name for all output files

print(f"Uploaded : {PDF_PATH}")
print(f"Doc ID   : {DOC_ID}")

## Step 1 — Marker PDF Parsing

Uses **Marker** (deep-learning PDF parser) to convert each slide into structured
markdown. For slides where Marker finds no text (image-only), a Tesseract OCR
fallback renders the page at 3× zoom and recovers text.

**Output:** `SlideMarkdown` per page — `slide_id`, `page_number`, `raw_markdown`, `doc_id`.

> First run downloads ~4 GB of Marker model weights — allow ~5 minutes.

In [ ]:
import hashlib, json, re
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import List

# ── Data model ────────────────────────────────────────────────────────────
@dataclass
class SlideMarkdown:
    slide_id:     str
    page_number:  int
    raw_markdown: str
    doc_id:       str

# ── Config ────────────────────────────────────────────────────────────────
PAGE_SEP = "-" * 48   # Marker page separator (must match converter config)

# ── Run Marker ────────────────────────────────────────────────────────────
print("Loading Marker models (first run downloads ~4 GB — allow ~5 min) ...")

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

model_dict = create_model_dict()
converter  = PdfConverter(
    artifact_dict=model_dict,
    config={"paginate_output": True, "page_separator": PAGE_SEP},
)
rendered      = converter(PDF_PATH)
full_markdown = rendered.markdown
raw_pages     = [p.strip() for p in full_markdown.split(PAGE_SEP)]
print(f"Marker produced {len(raw_pages)} raw page chunk(s)")

# ── OCR fallback for image-only pages ────────────────────────────────────
import fitz          # PyMuPDF
import pytesseract
from PIL import Image

def _is_empty_page(markdown: str) -> bool:
    """True when Marker found no real text on the page."""
    cleaned = markdown.strip()
    if cleaned in ["{0}", ""]:
        return True
    text_only = re.sub(r"!\[.*?\]\(.*?\)", "", cleaned).strip()
    text_only = re.sub(r"[#>\-\*\|]", "", text_only).strip()
    return len(text_only) < 10

def _ocr_page(pdf_path: str, page_number: int) -> str:
    """Render full page at 3× zoom and run Tesseract on it."""
    doc  = fitz.open(pdf_path)
    page = doc[page_number - 1]          # 0-indexed
    mat  = fitz.Matrix(3, 3)             # 3× zoom for accuracy
    pix  = page.get_pixmap(matrix=mat)
    img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = pytesseract.image_to_string(img, config="--psm 6")
    doc.close()
    return text.strip()

# ── Build SlideMarkdown list ───────────────────────────────────────────────
slides_md: List[SlideMarkdown] = []
ocr_count = 0

for idx, page_md in enumerate(raw_pages, start=1):
    page_md = page_md.strip()

    if _is_empty_page(page_md):
        print(f"  [slide_{idx:03d}] Marker: no text — running OCR ...")
        ocr_text = _ocr_page(PDF_PATH, idx)
        if ocr_text:
            # Prefix each OCR line with '> OCR:' so Step 2 can detect and
            # re-classify it properly instead of treating it as a caption.
            page_md = "\n".join(
                f"> OCR: {line}"
                for line in ocr_text.splitlines()
                if line.strip()
            )
            print(f"  [slide_{idx:03d}] OCR recovered {len(ocr_text):,} chars")
            ocr_count += 1
        else:
            print(f"  [slide_{idx:03d}] OCR found nothing — slide may be decorative, skipping")
            page_md = ""

    # Collapse excessive blank lines from Marker output
    page_md = re.sub(r"\n{3,}", "\n\n", page_md)

    if not page_md:
        continue

    slides_md.append(SlideMarkdown(
        slide_id    = f"slide_{idx:03d}",
        page_number = idx,
        raw_markdown= page_md,
        doc_id      = DOC_ID,
    ))

print(f"\nStep 1 complete: {len(slides_md)} slides ({ocr_count} used OCR fallback)")

# ── Save intermediate result ───────────────────────────────────────────────
STEP1_FILE = f"{STEM}_step1_parsed.json"
with open(STEP1_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(s) for s in slides_md], f, ensure_ascii=False, indent=2)
print(f"Saved  : {STEP1_FILE} ({Path(STEP1_FILE).stat().st_size:,} bytes)")

In [ ]:
# ── Step 1 inspection ─────────────────────────────────────────────────────
# Change SHOW_SLIDES and SHOW_FULL to inspect different pages.
SHOW_SLIDES = [1, 2, 3]   # page numbers to preview
SHOW_FULL   = False        # True = full markdown, False = first 500 chars

from IPython.display import Markdown, display

print(f"Total slides: {len(slides_md)}\n")
for slide in slides_md:
    if slide.page_number not in SHOW_SLIDES:
        continue
    display(Markdown(f"---\n### Slide {slide.page_number} — `{slide.slide_id}`\n---"))
    md = slide.raw_markdown
    preview = md if SHOW_FULL else (md[:500] + ("..." if len(md) > 500 else ""))
    print(preview)

In [ ]:
# ── Download Step 1 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP1_FILE)
print(f"Downloading {STEP1_FILE} ...")

## Step 2 — Markdown Preprocessor

Four stages applied to the raw markdown from Step 1:

1. **Structural parsing** — lines routed into typed buckets: headings, bullets, table cells, captions, body text
2. **Noise removal** — page numbers, copyright lines, URLs, image tags discarded
3. **Cross-slide repetition filter** — blocks appearing on >50% of slides removed (recurring headers/footers)
4. **Assembly** — buckets joined into `clean_text`; `heading_phrases` stored for Step 3 boost

OCR-recovered lines (prefixed `> OCR:`) are re-classified into proper buckets instead of being dumped into captions.

**Output:** `SlideContent` per slide.

In [ ]:
import re
from collections import Counter
from dataclasses import asdict, dataclass, field
from typing import List

# ── Data model ────────────────────────────────────────────────────────────
@dataclass
class SlideContent:
    slide_id:        str
    page_number:     int
    doc_id:          str
    headings:        List[str] = field(default_factory=list)
    body_text:       List[str] = field(default_factory=list)
    bullets:         List[str] = field(default_factory=list)
    table_cells:     List[str] = field(default_factory=list)
    captions:        List[str] = field(default_factory=list)
    code_lines:      List[str] = field(default_factory=list)
    heading_phrases: List[str] = field(default_factory=list)
    clean_text:      str       = ""
    ocr_applied:     bool      = False

# ── Noise patterns ────────────────────────────────────────────────────────
_NOISE_PATTERNS = [
    re.compile(r"^\s*page\s+\d+\s*(of\s+\d+)?\s*$", re.IGNORECASE),
    re.compile(r"^\s*\d+\s*$"),
    re.compile(r"^©.*", re.IGNORECASE),
    re.compile(r"^\s*(references|bibliography)\s*$", re.IGNORECASE),
    re.compile(r"https?://\S+"),
    re.compile(r"^\s*\[\d+\]"),
    re.compile(r"^!\[.*?\]\(.*?\)$"),           # image references ![]()
    re.compile(r"^quiz\s*:", re.IGNORECASE),
    # diagram legend labels (Input / Output / Legend / Infrastructure ...)
    re.compile(
        r"^(input|output|legend|infrastructure)(\s+replaced.*|\s+kept.*|\s+storage.*)?$",
        re.IGNORECASE,
    ),
]

def _is_noise(text: str) -> bool:
    t = text.strip()
    return any(p.match(t) for p in _NOISE_PATTERNS)

# ── Code-line detector ────────────────────────────────────────────────────
def _is_code_line(text: str) -> bool:
    """Structural signals — works across Python, Java, SQL, pseudocode, etc."""
    t = text.strip()
    if not t:
        return False
    if t.startswith("```"):
        return True
    code_chars = set("{}();=<>[]")
    code_char_count = sum(1 for c in t if c in code_chars)
    if len(t) > 5 and code_char_count / len(t) > 0.10:
        return True
    if re.search(r"\b[a-z]+[A-Z][a-zA-Z]+\b", t):    # camelCase
        return True
    if re.search(r"\b[a-z]+_[a-z]+\b", t):            # snake_case
        return True
    if re.search(r"\s(==|!=|<=|>=|&&|\|\||:=|->|=>)\s", t):  # operators
        return True
    if t.endswith((";", "{", "}")):                    # statement terminators
        return True
    if re.match(r"^\s*(//|#\s|--|/\*|\*\s)", t):      # comments
        return True
    if re.search(r"\b\w{2,}\(", t) and code_char_count >= 2:  # function call
        return True
    if re.match(r"^(default\s+)?constructor\s+\w+\s*\(", t, re.IGNORECASE):
        return True
    if re.match(r"^__\w+__\s*\(", t):                 # Python dunders
        return True
    return False

# ── Inline markdown stripper ──────────────────────────────────────────────
_INLINE_MD = re.compile(
    r"(\*\*|__)(.*?)\1"
    r"|(\*|_)(.*?)\3"
    r"|`([^`]+)`"
    r"|~~(.*?)~~"
    r"|\[([^\]]+)\]\([^)]+\)",
    re.DOTALL,
)

def _strip_inline(text: str) -> str:
    prev = None
    while prev != text:
        prev = text
        text = _INLINE_MD.sub(
            lambda m: next(g for g in m.groups() if g is not None
                           and g not in ("**", "__", "*", "_")),
            text,
        )
    return text.strip()

# ── OCR line re-classifier ────────────────────────────────────────────────
def _is_ocr_noise(text: str) -> bool:
    """Detect garbled OCR output."""
    t = text.strip()
    if not t or len(t) < 4:
        return True
    words = t.split()
    if len(words) == 1 and not words[0].isalpha():
        return True
    non_ascii = sum(1 for c in t if ord(c) > 127)
    if non_ascii / len(t) > 0.1:
        return True
    letters = sum(1 for c in t if c.isalpha())
    if len(t) > 5 and letters / len(t) < 0.4:
        return True
    return False

def _classify_ocr_line(text: str, sc: SlideContent) -> None:
    """Re-classify an OCR-recovered line into the proper bucket."""
    stripped = text.strip()
    if not stripped or _is_noise(stripped) or _is_ocr_noise(stripped):
        return
    if _is_code_line(stripped):
        sc.code_lines.append(stripped)
        return
    if stripped.startswith("#"):
        inner = re.sub(r"^#+\s*", "", stripped).strip()
        if inner:
            sc.headings.append(inner)
            sc.heading_phrases.append(inner)
    elif re.match(r"^[-*+]\s+", stripped):
        inner = re.sub(r"^[-*+]\s+", "", stripped).strip()
        if inner:
            sc.bullets.append(inner)
    elif stripped.startswith("|") and stripped.endswith("|"):
        if not re.match(r"^\|[\s\-\|]+\|$", stripped):
            cells = [c.strip() for c in stripped.split("|") if c.strip()]
            sc.table_cells.extend(cells)
    else:
        sc.body_text.append(stripped)

# ── Stage 1 + 2: parse single slide ───────────────────────────────────────
def _parse_slide(slide_md: SlideMarkdown) -> SlideContent:
    sc = SlideContent(
        slide_id    = slide_md.slide_id,
        page_number = slide_md.page_number,
        doc_id      = slide_md.doc_id,
    )
    for line in slide_md.raw_markdown.splitlines():
        s = line.strip()
        if not s:
            continue

        if s.startswith("#"):
            text = _strip_inline(re.sub(r"^#+\s*", "", s))
            if text and not _is_noise(text):
                sc.headings.append(text)
                sc.heading_phrases.append(text)

        elif re.match(r"^[-*]\s+", s) and not s.startswith(("**", "*a", "*b", "*c")):
            m = re.match(r"^[-*]\s+(.+)", s)
            text = _strip_inline(m.group(1)) if m else ""
            if text and not _is_noise(text):
                sc.bullets.append(text)

        elif s.startswith("|") and s.endswith("|"):
            cells = [_strip_inline(c.strip()) for c in s.split("|") if c.strip()]
            if all(re.fullmatch(r"[-:]+", c) for c in cells):
                continue
            sc.table_cells.extend(c for c in cells if c and not _is_noise(c))

        elif s.startswith(">"):
            inner = re.sub(r"^>\s*", "", s).strip()
            if inner.startswith("OCR:"):
                sc.ocr_applied = True
                _classify_ocr_line(inner[4:].strip(), sc)
            else:
                text = _strip_inline(inner)
                if text and not _is_noise(text):
                    sc.captions.append(text)

        else:
            text = _strip_inline(s)
            if not text or _is_noise(text):
                continue
            if _is_code_line(text):
                sc.code_lines.append(text)
            else:
                sc.body_text.append(text)

    return sc

# ── Stage 3: cross-slide repetition filter ────────────────────────────────
def _cross_slide_filter(contents: List[SlideContent]) -> List[SlideContent]:
    total = len(contents)
    if total < 5:
        print(f"Cross-slide filter skipped (only {total} slides — need 5+)")
        return contents
    threshold = total * 0.5
    counts: Counter = Counter()
    for sc in contents:
        seen = set()
        for bucket in (sc.headings, sc.body_text, sc.bullets, sc.table_cells, sc.captions):
            for t in bucket:
                if t not in seen:
                    counts[t] += 1
                    seen.add(t)
    noise = {t for t, c in counts.items() if c > threshold}
    if noise:
        print(f"Cross-slide filter removed {len(noise)} repeated block(s):")
        for n in sorted(noise):
            print(f"  '{n[:80]}'")
    else:
        print("Cross-slide filter: nothing removed")
    for sc in contents:
        sc.headings    = [t for t in sc.headings    if t not in noise]
        sc.body_text   = [t for t in sc.body_text   if t not in noise]
        sc.bullets     = [t for t in sc.bullets     if t not in noise]
        sc.table_cells = [t for t in sc.table_cells if t not in noise]
        sc.captions    = [t for t in sc.captions    if t not in noise]
    return contents

# ── Stage 4: assemble clean_text ──────────────────────────────────────────
def _assemble(sc: SlideContent) -> None:
    parts = sc.headings + sc.body_text + sc.bullets + sc.table_cells + sc.captions
    sc.clean_text = " ".join(parts).strip()

# ── Run all stages ────────────────────────────────────────────────────────
print("Stage 1+2: parsing and noise removal ...")
slide_contents: List[SlideContent] = [_parse_slide(s) for s in slides_md]

print("Stage 3: cross-slide repetition filter ...")
slide_contents = _cross_slide_filter(slide_contents)

print("Stage 4: assembling clean_text ...")
for sc in slide_contents:
    _assemble(sc)

# Filter out slides with no usable text
content_slides = [sc for sc in slide_contents if sc.clean_text.strip()]
print(f"\nStep 2 complete: {len(content_slides)}/{len(slide_contents)} slides have content")
print(f"  OCR applied on: {sum(1 for s in content_slides if s.ocr_applied)} slide(s)")

# ── Save intermediate result ───────────────────────────────────────────────
STEP2_FILE = f"{STEM}_step2_preprocessed.json"
with open(STEP2_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(s) for s in slide_contents], f, ensure_ascii=False, indent=2)
print(f"Saved  : {STEP2_FILE} ({Path(STEP2_FILE).stat().st_size:,} bytes)")

In [ ]:
# ── Step 2 inspection ─────────────────────────────────────────────────────
from IPython.display import Markdown, display

SHOW_SLIDES_2 = list(range(1, min(4, len(content_slides) + 1)))  # first 3 content slides

print(f"{'slide_id':<12} {'head':>4} {'body':>4} {'bull':>4} {'tbl':>4} {'cap':>4} {'code':>4} {'OCR':>4}  clean_text (first 120 chars)")
print("-" * 110)
for i, sc in enumerate(content_slides):
    flag = "Y" if sc.ocr_applied else "-"
    print(f"{sc.slide_id:<12} {len(sc.headings):>4} {len(sc.body_text):>4} "
          f"{len(sc.bullets):>4} {len(sc.table_cells):>4} {len(sc.captions):>4} "
          f"{len(sc.code_lines):>4} {flag:>4}  {sc.clean_text[:120]}")

print("\n" + "=" * 80)
print("Detailed view of first 3 content slides")
for sc in content_slides[:3]:
    display(Markdown(f"---\n### {sc.slide_id}  (page {sc.page_number})"))
    print(f"Headings    : {sc.headings}")
    print(f"Bullets     : {sc.bullets[:4]}")
    print(f"Body        : {sc.body_text[:2]}")
    print(f"Table cells : {sc.table_cells[:3]}")
    print(f"Captions    : {sc.captions[:2]}")
    print(f"Code lines  : {sc.code_lines[:2]}")
    print(f"OCR applied : {sc.ocr_applied}")
    print(f"clean_text  : {sc.clean_text[:300]}")

In [ ]:
# ── Download Step 2 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP2_FILE)
print(f"Downloading {STEP2_FILE} ...")

## Step 3 — Keyphrase Extraction (SIFRank-style)

Implements the **SIFRankSqueezeBERT** algorithm using:
- **spaCy `en_core_web_sm`** — candidate noun-chunk extraction and linguistic validation
- **`all-MiniLM-L6-v2`** (~80 MB) — phrase and document embeddings

Adaptive filter pipeline (in order):
1. Extract up to 30 noun-chunk candidates (code lines excluded)
2. Score by cosine similarity to document embedding
3. Quality threshold ≥ 0.3
4. spaCy linguistic filter (has noun, not all-stop, length ≥ 3)
5. Noun-chunk cross-validation
6. Assign source type + heading boost (+0.20)
7. Near-duplicate deduplication via SBERT (sim ≥ 0.85)

**Output:** `Keyphrase` list per slide — `phrase`, `score`, `source_type`, `appears_in`.

In [ ]:
import json, re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Dict

import spacy
from sentence_transformers import SentenceTransformer, util as st_util

# ── Config ────────────────────────────────────────────────────────────────
KEYPHRASE_MAX_CANDIDATES      = 30
KEYPHRASE_QUALITY_THRESHOLD   = 0.3
DEDUP_SIM_THRESHOLD           = 0.85

# ── Data model ────────────────────────────────────────────────────────────
@dataclass
class Keyphrase:
    phrase:      str
    score:       float
    source_type: str    # heading | body | bullet | table | caption
    slide_id:    str
    doc_id:      str
    appears_in:  str    # sentence containing the phrase

# ── Load models once ──────────────────────────────────────────────────────
print("Loading spaCy en_core_web_sm ...")
nlp = spacy.load("en_core_web_sm")
print("Loading all-MiniLM-L6-v2 (SIFRank embeddings) ...")
minilm = SentenceTransformer("all-MiniLM-L6-v2")
print("Models ready.")

# ── Phrase cleaner ────────────────────────────────────────────────────────
def _clean_phrase(phrase: str) -> str:
    if "```" in phrase:
        return ""
    phrase = re.sub(r"\{\d+\}", "", phrase)
    phrase = re.sub(r"\b\d+/\d+\b", "", phrase)
    phrase = re.sub(r"^(summary|overview|introduction|conclusion)\s+",
                    "", phrase, flags=re.IGNORECASE)
    return phrase.strip(" .,;:-/\\")

# ── Candidate extraction ──────────────────────────────────────────────────
def _extract_candidates(text: str) -> List[str]:
    doc  = nlp(text)
    seen: set = set()
    out: List[str] = []
    for chunk in doc.noun_chunks:
        phrase = re.sub(r"^(the|a|an)\s+", "", chunk.text.lower().strip())
        phrase = _clean_phrase(phrase)
        if not phrase or re.match(r"^\d", phrase) or len(phrase) < 3:
            continue
        if phrase not in seen:
            seen.add(phrase)
            out.append(phrase)
    return out

# ── SIFRank-style scoring ─────────────────────────────────────────────────
def _sifrank_scores(candidates: List[str], doc_text: str) -> Dict[str, float]:
    if not candidates or not doc_text.strip():
        return {}
    all_texts = [doc_text] + candidates
    embs = minilm.encode(all_texts, convert_to_tensor=True, show_progress_bar=False)
    doc_emb, cand_embs = embs[0], embs[1:]
    return {
        p: float(st_util.cos_sim(doc_emb.unsqueeze(0), e.unsqueeze(0)))
        for p, e in zip(candidates, cand_embs)
    }

# ── Linguistic filter ─────────────────────────────────────────────────────
def _is_valid(phrase: str) -> bool:
    doc = nlp(phrase)
    has_noun = any(t.pos_ in ("NOUN", "PROPN") for t in doc)
    all_stop = all(t.is_stop for t in doc)
    return has_noun and not all_stop and len(phrase.strip()) >= 3

def _in_noun_chunks(phrase: str, text: str) -> bool:
    doc    = nlp(text)
    chunks = {re.sub(r"^(the|a|an)\s+", "", c.text.lower().strip())
              for c in doc.noun_chunks}
    return phrase.lower() in chunks

# ── Source type + sentence finder ─────────────────────────────────────────
def _assign_source_type(phrase: str, sc: SlideContent) -> str:
    pl = phrase.lower()
    for h in sc.headings:
        if pl in h.lower(): return "heading"
    for b in sc.bullets:
        if pl in b.lower(): return "bullet"
    for t in sc.table_cells:
        if pl in t.lower(): return "table"
    for c in sc.captions:
        if pl in c.lower(): return "caption"
    return "body"

def _find_sentence(phrase: str, clean_text: str) -> str:
    doc = nlp(clean_text)
    pl  = phrase.lower()
    for sent in doc.sents:
        if pl in sent.text.lower():
            return sent.text.strip()
    return clean_text[:200]

# ── Near-duplicate deduplication ──────────────────────────────────────────
def _deduplicate(kps: List[Keyphrase]) -> List[Keyphrase]:
    if len(kps) <= 1:
        return kps
    sorted_kps = sorted(kps, key=lambda k: k.score, reverse=True)
    kept: List[Keyphrase] = []
    for kp in sorted_kps:
        emb = minilm.encode(kp.phrase, convert_to_tensor=True)
        is_dup = any(
            float(st_util.cos_sim(emb, minilm.encode(k.phrase, convert_to_tensor=True)))
            >= DEDUP_SIM_THRESHOLD
            for k in kept
        )
        if not is_dup:
            kept.append(kp)
    return kept

# ── Main extraction function ───────────────────────────────────────────────
def extract_keyphrases(sc: SlideContent) -> List[Keyphrase]:
    if not sc.clean_text.strip():
        return []

    # Build SIFRank input (exclude code lines)
    clean_body    = [l for l in sc.body_text if not _is_code_line(l)]
    clean_bullets = [b for b in sc.bullets   if not _is_code_line(b)]
    heading_text  = " ".join(sc.headings)
    rest_text     = " ".join(clean_body + clean_bullets + sc.table_cells + sc.captions)
    sifrank_text  = (heading_text + ". " + rest_text).strip() if heading_text and rest_text \
                    else (heading_text or rest_text)
    if not sifrank_text:
        return []

    candidates = _extract_candidates(sifrank_text)
    if not candidates:
        return []

    scores = _sifrank_scores(candidates, sifrank_text)

    # Quality threshold + cap
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top = [(p, s) for p, s in top[:KEYPHRASE_MAX_CANDIDATES]
           if s >= KEYPHRASE_QUALITY_THRESHOLD]

    # Linguistic + noun-chunk validation
    validated = [(p, s) for p, s in top
                 if _is_valid(p) and _in_noun_chunks(p, sifrank_text)]

    kps: List[Keyphrase] = []
    for phrase, score in validated:
        src    = _assign_source_type(phrase, sc)
        final  = min(score + 0.20, 1.0) if src == "heading" else score
        app_in = _find_sentence(phrase, sc.clean_text)
        kps.append(Keyphrase(
            phrase=phrase, score=round(final, 4), source_type=src,
            slide_id=sc.slide_id, doc_id=sc.doc_id, appears_in=app_in,
        ))

    kps = _deduplicate(kps)
    kps.sort(key=lambda k: k.score, reverse=True)
    return kps

# ── Run on all content slides ──────────────────────────────────────────────
print("Extracting keyphrases from all content slides ...")
keyphrases_by_slide: Dict[str, List[Keyphrase]] = {}
for i, sc in enumerate(content_slides, 1):
    kps = extract_keyphrases(sc)
    keyphrases_by_slide[sc.slide_id] = kps
    print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: {len(kps)} keyphrases")

total_kps = sum(len(v) for v in keyphrases_by_slide.values())
print(f"\nStep 3 complete: {total_kps} keyphrases across {len(content_slides)} slides")

# ── Save intermediate result ───────────────────────────────────────────────
STEP3_FILE = f"{STEM}_step3_keyphrases.json"
step3_out  = {sid: [asdict(k) for k in kps]
              for sid, kps in keyphrases_by_slide.items()}
with open(STEP3_FILE, "w", encoding="utf-8") as f:
    json.dump(step3_out, f, ensure_ascii=False, indent=2)
print(f"Saved  : {STEP3_FILE} ({Path(STEP3_FILE).stat().st_size:,} bytes)")

In [ ]:
# ── Step 3 inspection ─────────────────────────────────────────────────────
from IPython.display import Markdown, display

print(f"{'slide_id':<12}  {'#kps':>4}  top-5 keyphrases (phrase — score — source)")
print("-" * 100)
for sc in content_slides:
    kps = keyphrases_by_slide.get(sc.slide_id, [])
    top5 = ", ".join(
        f"{k.phrase} ({k.score:.2f}/{k.source_type[0]})"
        for k in kps[:5]
    )
    print(f"{sc.slide_id:<12}  {len(kps):>4}  {top5}")

In [ ]:
# ── Download Step 3 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP3_FILE)
print(f"Downloading {STEP3_FILE} ...")

## Step 4 — LLM Triple Extraction

**Core novel contribution.** Sends (keyphrases + slide text) to Groq's Llama 3 and
validates each returned triple through **3 mandatory layers**:

| Layer | Check |
|-------|-------|
| 1 — Anchor | subject AND object must be in the keyphrase list; relation must be one of 8 types |
| 2 — Evidence | evidence string must be semantically close to slide text (SBERT cosine ≥ 0.75) |
| 3 — Confidence | LLM-reported confidence ≥ 0.70 |

Enter your **Groq API key** below. Free at [console.groq.com](https://console.groq.com).

In [ ]:
import os
from getpass import getpass

# ── Groq API key ───────────────────────────────────────────────────────────
# Option A: paste key interactively (recommended — key not stored in notebook)
GROQ_API_KEY = getpass("Paste your Groq API key (hidden): ")

# Option B: set via environment variable before running:
#   os.environ['GROQ_API_KEY'] = 'gsk_...'
# Then comment out the getpass line above and uncomment:
#   GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

if not GROQ_API_KEY.strip():
    raise ValueError("GROQ_API_KEY is empty — cannot proceed with Step 4")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# ── Thresholds ─────────────────────────────────────────────────────────────
LLM_PRIMARY              = "llama-3.3-70b-versatile"    # best quality
LLM_FALLBACK             = "llama-3.1-8b-instant"       # faster, for rate-limit retries
TRIPLE_CONFIDENCE_THRESHOLD  = 0.70
EVIDENCE_SIMILARITY_THRESHOLD = 0.75

print("Config ready. API key accepted.")

In [ ]:
import json, time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Dict, Optional

from sentence_transformers import SentenceTransformer, util as st_util
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

# ── Data model ────────────────────────────────────────────────────────────
@dataclass
class Triple:
    subject:    str
    relation:   str
    object:     str
    evidence:   str     # exact sentence copied from slide text
    confidence: float
    slide_id:   str
    doc_id:     str
    source:     str = "extraction"

# ── Prompt templates (verbatim from CLAUDE.md) ────────────────────────────
SYSTEM_PROMPT = """You are a knowledge graph construction assistant for educational materials.

STRICT RULES — violating any rule invalidates your entire response:
1. ONLY use concepts from the KEYPHRASE LIST as subject and object. Nothing else.
2. Every triple MUST include an EXACT sentence copied from the SLIDE TEXT as evidence.
3. If no supporting sentence exists in the slide text, omit that triple entirely.
4. Use ONLY these relation types:
   isPrerequisiteOf   - A must be understood before B
   isDefinedAs        - formal definition of A is given
   isExampleOf        - A is a specific example of B
   contrastedWith     - A and B are explicitly compared
   appliedIn          - A is used/applied in context B
   isPartOf           - A is a structural component of B
   causeOf            - A causes or leads to B
   isGeneralizationOf - A is a broader category including B
5. Return ONLY a valid JSON array. No markdown, no explanation, no preamble.
6. If no valid triples found, return: []"""

USER_PROMPT = """KEYPHRASE LIST (subject and object MUST come from this list):
{keyphrases}

SLIDE TEXT:
{slide_text}

Extract all valid triples as JSON array. Each item:
{{
  "subject": "exact phrase from keyphrase list",
  "relation": "one of the 8 relation types",
  "object": "exact phrase from keyphrase list (different from subject)",
  "evidence": "exact sentence copied from slide text above",
  "confidence": 0.0 to 1.0
}}"""

# ── LLM clients ───────────────────────────────────────────────────────────
_primary_llm  = None
_fallback_llm = None

def _get_primary() -> ChatGroq:
    global _primary_llm
    if _primary_llm is None:
        _primary_llm = ChatGroq(
            model=LLM_PRIMARY,
            temperature=0,
            model_kwargs={"response_format": {"type": "json_object"}},
        )
    return _primary_llm

def _get_fallback() -> ChatGroq:
    global _fallback_llm
    if _fallback_llm is None:
        _fallback_llm = ChatGroq(
            model=LLM_FALLBACK,
            temperature=0,
            model_kwargs={"response_format": {"type": "json_object"}},
        )
    return _fallback_llm

def _invoke_with_fallback(messages: list, max_retries: int = 3) -> Optional[str]:
    """Try primary LLM; on HTTP 429 fall back to smaller model then wait."""
    for attempt in range(1, max_retries + 1):
        try:
            return str(_get_primary().invoke(messages).content)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                print(f"    Rate-limited (attempt {attempt}) — trying fallback ...")
                try:
                    return str(_get_fallback().invoke(messages).content)
                except Exception as e2:
                    if "429" in str(e2) or "rate" in str(e2).lower():
                        print(f"    Fallback also rate-limited — waiting 5 s ...")
                        time.sleep(5)
                        continue
                    print(f"    Fallback error: {e2}")
                    return None
            print(f"    LLM error: {e}")
            return None
    return None

def _parse_llm_response(content: str) -> List[dict]:
    if not content:
        return []
    try:
        raw = json.loads(content)
    except json.JSONDecodeError:
        return []
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        for v in raw.values():
            if isinstance(v, list):
                return v
    return []

# ── Load SBERT for Layer 2 evidence verification ───────────────────────────
print("Loading all-mpnet-base-v2 for evidence verification (SBERT) ...")
sbert = SentenceTransformer("all-mpnet-base-v2")
print("SBERT ready.")

_VALID_RELATIONS = frozenset({
    "isPrerequisiteOf", "isDefinedAs", "isExampleOf", "contrastedWith",
    "appliedIn", "isPartOf", "causeOf", "isGeneralizationOf",
})

# ── 3-layer validator ─────────────────────────────────────────────────────
def validate_triple(
    raw: dict,
    kp_set: set,          # lowercase keyphrase phrases
    slide_text: str,
    slide_id: str,
    doc_id: str,
) -> Optional[Triple]:
    subject  = str(raw.get("subject",   "")).lower().strip()
    obj      = str(raw.get("object",    "")).lower().strip()
    relation = str(raw.get("relation",  "")).strip()
    evidence = str(raw.get("evidence",  "")).strip()
    try:
        conf = float(raw.get("confidence", 0))
    except (TypeError, ValueError):
        conf = 0.0

    # Layer 1: anchor constraint
    if subject not in kp_set or obj not in kp_set:
        return None
    if subject == obj:
        return None
    if relation not in _VALID_RELATIONS:
        return None

    # Layer 2: evidence verification
    if not evidence:
        return None
    ev_emb   = sbert.encode(evidence,   convert_to_tensor=True, show_progress_bar=False)
    text_emb = sbert.encode(slide_text, convert_to_tensor=True, show_progress_bar=False)
    sim = float(st_util.cos_sim(ev_emb, text_emb))
    if sim < EVIDENCE_SIMILARITY_THRESHOLD:
        return None

    # Layer 3: confidence threshold
    if conf < TRIPLE_CONFIDENCE_THRESHOLD:
        return None

    return Triple(
        subject=subject, relation=relation, object=obj,
        evidence=evidence, confidence=conf,
        slide_id=slide_id, doc_id=doc_id,
    )

# ── Run extraction on all content slides ──────────────────────────────────
print("\nExtracting triples (1 LLM call per slide) ...")
all_triples: List[Triple] = []
stats = {"slides": 0, "skipped": 0, "raw": 0, "passed": 0}

for i, sc in enumerate(content_slides, 1):
    kps = keyphrases_by_slide.get(sc.slide_id, [])
    if not kps or not sc.clean_text.strip():
        print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: skipped (no keyphrases or text)")
        stats["skipped"] += 1
        continue

    kp_list = "\n".join(f"- {k.phrase}" for k in kps)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=USER_PROMPT.format(
            keyphrases=kp_list,
            slide_text=sc.clean_text,
        )),
    ]

    content = _invoke_with_fallback(messages)
    raw_dicts = _parse_llm_response(content or "")
    stats["raw"] += len(raw_dicts)

    kp_set = {k.phrase.lower() for k in kps}
    slide_triples: List[Triple] = []
    for rd in raw_dicts:
        t = validate_triple(rd, kp_set, sc.clean_text, sc.slide_id, sc.doc_id)
        if t is not None:
            slide_triples.append(t)

    all_triples.extend(slide_triples)
    stats["slides"] += 1
    stats["passed"] += len(slide_triples)

    print(f"  [{i:2d}/{len(content_slides)}] {sc.slide_id}: "
          f"{len(raw_dicts)} raw → {len(slide_triples)} valid triple(s)")

print(f"\nStep 4 complete:")
print(f"  Slides processed : {stats['slides']} (skipped: {stats['skipped']})")
print(f"  Raw candidates   : {stats['raw']}")
print(f"  Validated triples: {stats['passed']} "
      f"({100*stats['passed']/stats['raw']:.0f}% pass rate)"
      if stats['raw'] else "  Validated triples: 0")

# ── Save intermediate result ───────────────────────────────────────────────
STEP4_FILE = f"{STEM}_step4_triples.json"
with open(STEP4_FILE, "w", encoding="utf-8") as f:
    json.dump([asdict(t) for t in all_triples], f, ensure_ascii=False, indent=2)
print(f"Saved  : {STEP4_FILE} ({Path(STEP4_FILE).stat().st_size:,} bytes)")

In [ ]:
# ── Step 4 inspection ─────────────────────────────────────────────────────
from collections import Counter
from IPython.display import Markdown, display

rel_counts = Counter(t.relation for t in all_triples)

print(f"Total validated triples : {len(all_triples)}")
print(f"\nRelation type distribution:")
for rel, cnt in rel_counts.most_common():
    bar = "█" * cnt
    print(f"  {rel:<22} {cnt:>3}  {bar}")

print("\n" + "=" * 80)
print("Sample triples (first 10):")
for t in all_triples[:10]:
    display(Markdown(
        f"**[{t.slide_id}]** `{t.subject}` —*{t.relation}*→ `{t.object}`  "
        f"conf={t.confidence:.2f}  "
        f"\n> *{t.evidence[:120]}*"
    ))

In [ ]:
# ── Download Step 4 output ─────────────────────────────────────────────────
from google.colab import files as _files
_files.download(STEP4_FILE)
print(f"Downloading {STEP4_FILE} ...")

# ── Download all intermediate files at once (optional) ────────────────────
print("\nDownload all intermediate files:")
for fname in [STEP1_FILE, STEP2_FILE, STEP3_FILE, STEP4_FILE]:
    _files.download(fname)
    print(f"  {fname}")
print("Done — check your browser Downloads folder.")